# masarch 0.1.0 导入与快速使用

这个 notebook 解决两个最容易混淆的问题：

- **安装名**是 `masarch`
- **导入名**是 `agentorch`

也就是说，如果你要安装 `0.1.0`，命令是：

```bash
pip install masarch==0.1.0
```

但在 Python / Jupyter 里导入时，应该写：

```python
import agentorch
```

说明：

- 这个 notebook 默认**优先使用真实模型**，如果当前内核没有配置好，就会明确提示缺了什么。
- 只有在你自己把 `use_dummy` 设成 `True` 时，才会走离线演示。
- 在 Jupyter 里请优先使用 `await agent.run(...)`，不要默认用 `run_sync()`。

In [ ]:
# 如果你是在全新 notebook 内核里第一次使用，可以先运行这一格。
# 已经装过就不用重复执行。

# %pip install -U pip
# %pip install masarch==0.1.0

In [5]:
from __future__ import annotations

import importlib.metadata as metadata
import os
import subprocess
import sys
from pathlib import Path

import agentorch

NOTEBOOK_PROJECT_ROOT = Path.cwd()
NOTEBOOK_ENV_PATH = NOTEBOOK_PROJECT_ROOT / ".env"
NOTEBOOK_ENV_EXAMPLE_PATH = NOTEBOOK_PROJECT_ROOT / ".env.example"

print("导入成功：", agentorch.__name__)
print("导入文件：", agentorch.__file__)
print("当前内核解释器：", sys.executable)

dist = None
try:
    dist = metadata.distribution("masarch")
    print("已安装的 masarch 版本：", dist.version)
    print("masarch 元数据路径：", dist.locate_file(""))
except metadata.PackageNotFoundError:
    print("当前内核没有 masarch 的包元数据。")
    print("这通常说明两种情况之一：")
    print("1. 当前内核是从源码目录直接 import agentorch")
    print("2. masarch 装在别的 Python / 别的 Jupyter 内核里")

pip_show = subprocess.run(
    [sys.executable, "-m", "pip", "show", "masarch"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
if pip_show.returncode == 0 and pip_show.stdout.strip():
    print("当前内核对应 pip show masarch：")
    print(pip_show.stdout.strip())
else:
    print("当前内核对应 pip 未发现 masarch。")

print("当前工作目录：", NOTEBOOK_PROJECT_ROOT)
print(".env 是否存在：", NOTEBOOK_ENV_PATH.exists())
print(".env.example 是否存在：", NOTEBOOK_ENV_EXAMPLE_PATH.exists())
print("结论：安装名是 masarch，导入名是 agentorch")
if dist is None:
    print("当前 notebook 仍可从源码目录导入 agentorch，但这不等于当前内核已经安装 masarch。")
print("如果环境变量齐全，下面会优先加载真实 LLM。")


导入成功： agentorch
导入文件： c:\Users\24260\Desktop\研究生生涯\agentorch\agentorch\__init__.py
当前内核解释器： c:\Users\24260\.conda\envs\data_analysis_py311\python.exe
已安装的 masarch 版本： 0.1.0
masarch 元数据路径： c:\Users\24260\.conda\envs\data_analysis_py311\Lib\site-packages
当前内核对应 pip show masarch：
Name: masarch
Version: 0.1.0
Summary: Code-first async-first agent orchestration framework for Python.
Home-page: https://github.com/Akun-python/agentorch
Author: 
Author-email: Guangpeng Sun <2303907916@qq.com>, Haojie Sun <2927535815@qq.com>
License-Expression: MIT
Location: c:\Users\24260\.conda\envs\data_analysis_py311\Lib\site-packages
Requires: httpx, jinja2, openai, pydantic
Required-by:
当前工作目录： c:\Users\24260\Desktop\研究生生涯\agentorch
.env 是否存在： True
.env.example 是否存在： True
结论：安装名是 masarch，导入名是 agentorch
如果环境变量齐全，下面会优先加载真实 LLM。


In [6]:
# 优先顺序：
# 1. 当前内核已经存在的环境变量
# 2. 仓库根目录下的 .env（需要显式加载，只补当前内核里缺失的值）
# notebook 本身不再写入 OPENAI_* 变量，避免覆盖真实环境配置。

LOAD_DOTENV = True

if LOAD_DOTENV and NOTEBOOK_ENV_PATH.exists():
    agentorch.initialize_environment(NOTEBOOK_ENV_PATH, overwrite=False)
    print("已从 .env 加载环境变量：", NOTEBOOK_ENV_PATH.name)
else:
    print("未从 .env 加载：", NOTEBOOK_ENV_PATH.name if not NOTEBOOK_ENV_PATH.exists() else "LOAD_DOTENV=False")

model_aliases = {
    "OPENAI_CHAT_MODEL": os.getenv("OPENAI_CHAT_MODEL"),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "AGENTORCH_MODEL": os.getenv("AGENTORCH_MODEL"),
}
print("OPENAI_API_KEY 已设置：", bool(os.getenv("OPENAI_API_KEY")))
print("模型别名：", model_aliases)
print("OPENAI_MODEL：", model_aliases["OPENAI_CHAT_MODEL"] or model_aliases["OPENAI_MODEL"] or model_aliases["AGENTORCH_MODEL"])
print("OPENAI_BASE_URL：", os.getenv("OPENAI_BASE_URL"))


已从 .env 加载环境变量： .env
OPENAI_API_KEY 已设置： True
模型别名： {'OPENAI_CHAT_MODEL': 'deepseek-v4-flash', 'OPENAI_MODEL': 'deepseek-v4-flash', 'AGENTORCH_MODEL': 'deepseek-v4-flash'}
OPENAI_MODEL： deepseek-v4-flash
OPENAI_BASE_URL： https://www.dmxapi.cn/v1


## 1. 真实模型优先的最小示例

这一段先尝试读取当前内核环境里的真实模型配置：

- `OPENAI_API_KEY`
- `OPENAI_CHAT_MODEL` / `OPENAI_MODEL` / `AGENTORCH_MODEL`
- 可选 `OPENAI_BASE_URL`

如果这些变量齐全，就直接加载真实 LLM；如果缺失，就明确打印缺了什么，并保留一个可切换的离线演示分支。

In [7]:
from agentorch.config import ModelConfig


def build_live_model_or_none():
    api_key = os.getenv("OPENAI_API_KEY")
    model_name = os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL")
    base_url = os.getenv("OPENAI_BASE_URL")
    missing = []
    if not api_key:
        missing.append("OPENAI_API_KEY")
    if not model_name:
        missing.append("OPENAI_CHAT_MODEL / OPENAI_MODEL / AGENTORCH_MODEL")
    if missing:
        return None, missing
    return agentorch.OpenAIModel(api_key=api_key, base_url=base_url, model=model_name), []


live_model, missing_items = build_live_model_or_none()
if live_model is None:
    print("未加载真实 LLM，缺少：", ", ".join(missing_items))
    print("如需离线演示，把 use_dummy 改成 True。")
else:
    print("真实模型已加载：", live_model.__class__.__name__)
    config_preview = ModelConfig.from_any({
        "api_key": os.getenv("OPENAI_API_KEY"),
        "base_url": os.getenv("OPENAI_BASE_URL"),
        "model": os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL"),
    }).model_dump()
    for key, value in list(config_preview.items()):
        if key.endswith("api_key") and value:
            config_preview[key] = str(value)[:6] + "..."
    print(
        "模型配置：",
        config_preview,
    )


from agentorch.core import Message, ModelRequest, ModelResponse, UsageInfo
from agentorch.models.base import BaseModelAdapter


class DummyModel(BaseModelAdapter):
    def __init__(self, *, name: str = "dummy-model", reply: str = "hello from masarch") -> None:
        self.config = {"provider": "dummy", "api_key": "sk-dummy", "model": name}
        self.reply = reply
        self.closed = False

    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role="assistant", content=self.reply),
            content=self.reply,
            finish_reason="stop",
            usage=UsageInfo(total_tokens=5),
        )

    async def aclose(self) -> None:
        self.closed = True

真实模型已加载： OpenAIModel
模型配置： {'provider': 'openai', 'model': 'deepseek-v4-flash', 'vision_model': None, 'api_key': 'sk-8G2...', 'base_url': 'https://www.dmxapi.cn/v1', 'endpoint_path': '/chat/completions', 'auth_scheme': 'Bearer', 'headers': {}, 'embedding_api_key': 'sk-8G2...', 'embedding_base_url': 'https://www.dmxapi.cn/v1', 'embedding_endpoint_path': '/embeddings', 'embedding_model': None, 'embedding_dimensions': None, 'speech_api_key': 'sk-8G2...', 'speech_base_url': 'https://www.dmxapi.cn/v1', 'speech_endpoint_path': '/audio/speech', 'speech_model': None, 'speech_voice': None, 'speech_format': 'mp3', 'speech_speed': 1.0, 'image_api_key': 'sk-8G2...', 'image_base_url': 'https://api.apiyi.com/v1', 'image_explicit_url': None, 'image_model': 'gemini-2.5-pro', 'image_aspect_ratio': '16:9', 'image_size': '2K', 'image_timeout': 300.0, 'image_fallback_models': [], 'image_retry_without_proxy': True, 'image_disable_env_proxy': False, 'video_api_key': 'sk-8G2...', 'video_base_url': 'https://w

In [8]:
use_dummy = False  # 默认关闭离线演示，避免把假模型误当真实 LLM
selected_model = live_model if live_model is not None else DummyModel(reply="minimal agent ok")
agent = agentorch.create_agent(
    model=selected_model,
    system_prompt="You are concise.",
    reasoning="react",
)

if live_model is None and not use_dummy:
    print("当前不会执行离线 DummyModel，因为你要的是实时 LLM 优先。")
else:
    result = await agent.run(
        "请回复一句最短确认语。",
        thread_id="nb-import-quickstart-001",
    )

    print("输出：", result.output_text)
    print("tokens：", result.usage.total_tokens)
    print("blueprint kind：", agent.export_blueprint()["kind"])

    await agent.aclose()


输出： 好
tokens： 150
blueprint kind： single_agent


## 2. 工具调用示例

这个示例说明导入后不仅能创建 agent，也能挂载工具。

In [9]:
use_dummy = False  # 默认不回退到离线演示，避免把假模型误当真实 LLM
from pydantic import BaseModel
from agentorch import ToolRegistry, tool

class AddInput(BaseModel):
    a: int
    b: int

@tool(description="Add two integers.")
async def add_numbers(input: AddInput):
    return {"sum": input.a + input.b}

if live_model is None and not use_dummy:
    print("工具示例跳过：当前没有真实 LLM；如需离线演示，把 use_dummy 改成 True。")
else:
    tool_agent = agentorch.create_agent(
        model=(live_model if live_model is not None else DummyModel(reply="tool agent ok")),
        tools=ToolRegistry.from_tools(add_numbers),
        reasoning="react",
    )

    tool_result = await tool_agent.run(
        "Use add_numbers to compute 12 + 30.",
        thread_id="nb-import-quickstart-002",
    )

    print("模型输出：", tool_result.output_text)
    print("tool_results 数量：", len(tool_result.tool_results))

    await tool_agent.aclose()


模型输出： The result of 12 + 30 is **42**.
tool_results 数量： 1


In [10]:
# 这个空白单元预留给后续扩展示例。


## 2.5 模型客制化与多供应商接入

这一节专门说明：**这个库如何自定义 model，以及如何给每个 model / 能力单独配置 `api_key`、`base_url`。**

当前库已经内建了几条常用路径：

1. `create_agent(model="模型名")`：模型名走环境变量，适合单供应商默认配置。
2. `agentorch.OpenAIModel(...)`：显式传入聊天模型、视觉模型、embedding、语音、图片、视频等各自的配置。
3. `create_agent(model={...})`：直接传 provider 配置字典，由库内部工厂创建适配器。
4. `agentorch.OpenAICompatibleHTTPModel(...)`：接入任意 OpenAI 兼容 HTTP 服务。
5. `agentorch.register_model_provider(...)`：注册你自己的 provider 名称和构造逻辑。

下面几格分别演示这几种方式。

In [11]:
import agentorch
from agentorch import OpenAICompatibleHTTPModel, OpenAIModel
from agentorch.config import ModelConfig


def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
    if not value:
        return None
    value = str(value)
    return value[:keep] + "..." if len(value) > keep else value


def preview_model_config(config: ModelConfig | dict | None) -> dict:
    payload = ModelConfig.from_any(config).model_dump()
    for key, value in list(payload.items()):
        if key.endswith("api_key") and value:
            payload[key] = mask_secret(value)
    return payload


print("已注册 provider：", agentorch.list_model_providers())


已注册 provider： ['openai', 'openai_http']


### 方式 A：只传模型名，默认走环境变量

这是最省事的方式，适合：

- 当前 notebook / 进程只接一个默认聊天供应商
- `OPENAI_API_KEY`、`OPENAI_BASE_URL` 已经在 `.env` 或 notebook 配置单元里设置好

注意：这里的 `model="deepseek-v4-flash"` 只提供模型名，真正的 `api_key/base_url` 仍来自环境变量。

In [12]:
import agentorch
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

default_model_config = ModelConfig.from_any("deepseek-v4-flash")
print("默认环境变量驱动的 ModelConfig：")
print(preview_model_config(default_model_config))

default_agent = agentorch.create_agent(
    model="deepseek-v4-flash",
    system_prompt="你是一个简洁的中文助手。",
    reasoning="react",
    name="default-env-agent",
)
print("default_agent blueprint kind：", default_agent.export_blueprint()["kind"])
await default_agent.aclose()


默认环境变量驱动的 ModelConfig：
{'provider': 'openai', 'model': 'deepseek-v4-flash', 'vision_model': None, 'api_key': 'sk-8G2...', 'base_url': 'https://www.dmxapi.cn/v1', 'endpoint_path': '/chat/completions', 'auth_scheme': 'Bearer', 'headers': {}, 'embedding_api_key': 'sk-n6p...', 'embedding_base_url': 'https://www.dmxapi.cn/v1', 'embedding_endpoint_path': '/embeddings', 'embedding_model': None, 'embedding_dimensions': None, 'speech_api_key': 'sk-8G2...', 'speech_base_url': 'https://www.dmxapi.cn/v1', 'speech_endpoint_path': '/audio/speech', 'speech_model': None, 'speech_voice': None, 'speech_format': 'mp3', 'speech_speed': 1.0, 'image_api_key': 'sk-Yja...', 'image_base_url': 'https://api.apiyi.com/v1', 'image_explicit_url': None, 'image_model': 'gemini-2.5-pro', 'image_aspect_ratio': '16:9', 'image_size': '2K', 'image_timeout': 300.0, 'image_fallback_models': [], 'image_retry_without_proxy': True, 'image_disable_env_proxy': False, 'video_api_key': 'sk-8G2...', 'video_base_url': 'https://www.d

### 方式 B：显式构造单个聊天模型

当你不想让聊天模型依赖当前环境，而是想为某一个 agent 明确指定：

- 模型名
- `api_key`
- `base_url`

就直接构造 `OpenAIModel(...)`。这条路径最适合给单个 agent 绑定独立 LLM。

In [13]:
import os
from agentorch import OpenAIModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

custom_chat_model = OpenAIModel(
    model="deepseek-v4-flash",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    timeout=45.0,
    max_retries=1,
)
print("显式聊天模型配置：")
print(preview_model_config(custom_chat_model.config))
await custom_chat_model.aclose()


显式聊天模型配置：
{'provider': 'openai', 'model': 'deepseek-v4-flash', 'vision_model': None, 'api_key': 'sk-8G2...', 'base_url': 'https://www.dmxapi.cn/v1', 'endpoint_path': '/chat/completions', 'auth_scheme': 'Bearer', 'headers': {}, 'embedding_api_key': 'sk-8G2...', 'embedding_base_url': 'https://www.dmxapi.cn/v1', 'embedding_endpoint_path': '/embeddings', 'embedding_model': None, 'embedding_dimensions': None, 'speech_api_key': 'sk-8G2...', 'speech_base_url': 'https://www.dmxapi.cn/v1', 'speech_endpoint_path': '/audio/speech', 'speech_model': None, 'speech_voice': None, 'speech_format': 'mp3', 'speech_speed': 1.0, 'image_api_key': 'sk-8G2...', 'image_base_url': 'https://api.apiyi.com/v1', 'image_explicit_url': None, 'image_model': 'gemini-2.5-pro', 'image_aspect_ratio': '16:9', 'image_size': '2K', 'image_timeout': 300.0, 'image_fallback_models': [], 'image_retry_without_proxy': True, 'image_disable_env_proxy': False, 'video_api_key': 'sk-8G2...', 'video_base_url': 'https://www.dmxapi.cn/v1',

### 方式 C：为 embedding / speech / image / video 分别配置

这是当前库最实用的一条能力：**你可以让聊天、embedding、图片、视频分别走不同的 key 和 base_url。**

例如：

- 聊天走 `dmxapi`
- 图片走 `apiyi`
- embedding 走另一家兼容接口

只要在 `OpenAIModel(...)` 里分别传对应字段即可。

In [ ]:
import os
from agentorch import OpenAIModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

multi_capability_model = OpenAIModel(
    model="deepseek-v4-flash",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    embedding_api_key=os.getenv("OPENAI_EMBEDDING_API_KEY") or os.getenv("OPENAI_API_KEY"),
    embedding_base_url=os.getenv("OPENAI_EMBEDDING_BASE_URL") or os.getenv("OPENAI_BASE_URL"),
    embedding_model=os.getenv("OPENAI_EMBEDDING_MODEL") or "text-embedding-3-small",
    image_api_key=os.getenv("OPENAI_IMAGE_API_KEY"),
    image_base_url=os.getenv("OPENAI_IMAGE_BASE_URL"),
    image_model=os.getenv("OPENAI_IMAGE_MODEL") or "gemini-2.5-pro",
    video_api_key=os.getenv("OPENAI_VIDEO_API_KEY") or os.getenv("OPENAI_API_KEY"),
    video_base_url=os.getenv("OPENAI_VIDEO_BASE_URL") or os.getenv("OPENAI_BASE_URL"),
    video_model=os.getenv("OPENAI_VIDEO_MODEL"),
)
print("按能力拆分后的模型配置：")
print(preview_model_config(multi_capability_model.config))
await multi_capability_model.aclose()


按能力拆分后的模型配置：
{'provider': 'openai', 'model': 'deepseek-v4-flash', 'vision_model': None, 'api_key': 'sk-8G2...', 'base_url': 'https://www.dmxapi.cn/v1', 'endpoint_path': '/chat/completions', 'auth_scheme': 'Bearer', 'headers': {}, 'embedding_api_key': 'sk-n6p...', 'embedding_base_url': 'https://www.dmxapi.cn/v1', 'embedding_endpoint_path': '/embeddings', 'embedding_model': 'text-embedding-3-small', 'embedding_dimensions': None, 'speech_api_key': 'sk-8G2...', 'speech_base_url': 'https://www.dmxapi.cn/v1', 'speech_endpoint_path': '/audio/speech', 'speech_model': None, 'speech_voice': None, 'speech_format': 'mp3', 'speech_speed': 1.0, 'image_api_key': 'sk-Yja...', 'image_base_url': 'https://api.apiyi.com/v1', 'image_explicit_url': None, 'image_model': 'gemini-2.5-pro', 'image_aspect_ratio': '16:9', 'image_size': '2K', 'image_timeout': 300.0, 'image_fallback_models': [], 'image_retry_without_proxy': True, 'image_disable_env_proxy': False, 'video_api_key': 'sk-8G2...', 'video_base_url': 'htt

### 方式 C.1：embedding 模型示例

下面这个示例直接调用 `embed_text(...)`，适合验证：

- `embedding_model` 是否配置正确
- `embedding_base_url` 是否真的指向 `/v1` 或 provider 根地址
- 返回向量维度是否符合预期


In [15]:
import os
from agentorch import OpenAIModel

embedding_model_name = os.getenv("OPENAI_EMBEDDING_MODEL") or "text-embedding-3-small"
embedding_base_url = os.getenv("OPENAI_EMBEDDING_BASE_URL") or os.getenv("OPENAI_BASE_URL")
embedding_api_key = os.getenv("OPENAI_EMBEDDING_API_KEY") or os.getenv("OPENAI_API_KEY")

if not embedding_api_key or not embedding_base_url:
    print("跳过 embedding 示例：缺少 OPENAI_EMBEDDING_API_KEY / OPENAI_API_KEY 或 embedding base_url。")
else:
    embedding_demo_model = OpenAIModel(
        model=os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL"),
        embedding_api_key=embedding_api_key,
        embedding_base_url=embedding_base_url,
        embedding_model=embedding_model_name,
    )
    vector = await embedding_demo_model.embed_text("AgentTorch 让多智能体编排更简单。")
    print("embedding 模型：", embedding_model_name)
    print("embedding base_url：", embedding_base_url)
    print("向量维度：", len(vector))
    print("前 8 个值：", [round(value, 6) for value in vector[:8]])
    await embedding_demo_model.aclose()


AuthenticationError: Error code: 401 - {'error': {'code': '', 'message': 'Invalid Token (request id: 20260507133345403198392gBd7RNL2)', 'type': 'rix_api_error'}}

### 方式 C.2：图片理解模型示例

这个示例演示 `analyze_image(...)`。默认读本仓库里的 `resource/agentorch-icon.svg`，你也可以改成自己的 PNG / JPG。

建议优先保证：

- `vision_model` 或聊天模型本身支持图片输入
- `image_path` 指向真实存在的文件
- 如果你接的是兼容接口，`OPENAI_BASE_URL` 保持在 provider 根地址


In [2]:
import os
from pathlib import Path
from agentorch import OpenAIModel

image_demo_path = NOTEBOOK_PROJECT_ROOT / "resource" / "agentorch-icon.svg"
vision_model_name = os.getenv("OPENAI_VISION_MODEL") or os.getenv("OPENAI_IMAGE_UNDERSTANDING_MODEL") or os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL")

if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_BASE_URL"):
    print("跳过图片理解示例：缺少 OPENAI_API_KEY 或 OPENAI_BASE_URL。")
elif not vision_model_name:
    print("跳过图片理解示例：缺少可用模型名。")
elif not image_demo_path.exists():
    print("跳过图片理解示例：图片文件不存在 ->", image_demo_path)
else:
    image_demo_model = OpenAIModel(
        model=vision_model_name,
        vision_model=vision_model_name,
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL"),
    )
    image_result = await image_demo_model.analyze_image(
        prompt="请用中文描述这张图片里的主体、风格和可能的品牌含义。",
        image_path=image_demo_path,
        model=vision_model_name,
        max_tokens=300,
    )
    print("图片理解模型：", vision_model_name)
    print("图片路径：", image_demo_path)
    print("模型输出：", image_result.content)
    await image_demo_model.aclose()


NameError: name 'NOTEBOOK_PROJECT_ROOT' is not defined

### 方式 C.3：视频理解模型示例

这个示例调用 `analyze_video(...)`。为了避免默认引用一个并不存在的视频文件，这里要求你先准备本地视频，或先设置环境变量 `NOTEBOOK_VIDEO_PATH`。

推荐视频格式：

- `.mp4`
- 时长先控制在几十秒内，便于快速验证
- 内容尽量单一，先验证接口链路，再测复杂场景


In [ ]:
import os
from pathlib import Path
from agentorch import OpenAIModel

video_model_name = os.getenv("OPENAI_VIDEO_MODEL") or os.getenv("OPENAI_VISION_MODEL") or os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL")
video_path_value = os.getenv("NOTEBOOK_VIDEO_PATH", "").strip()
video_demo_path = Path(video_path_value).expanduser() if video_path_value else None
video_base_url = os.getenv("OPENAI_VIDEO_BASE_URL") or os.getenv("OPENAI_BASE_URL")
video_api_key = os.getenv("OPENAI_VIDEO_API_KEY") or os.getenv("OPENAI_API_KEY")

if not video_path_value:
    print("跳过视频理解示例：请先设置 NOTEBOOK_VIDEO_PATH 指向本地 mp4 文件。")
elif video_demo_path is None or not video_demo_path.exists():
    print("跳过视频理解示例：视频文件不存在 ->", video_demo_path)
elif not video_api_key or not video_base_url:
    print("跳过视频理解示例：缺少 OPENAI_VIDEO_API_KEY / OPENAI_API_KEY 或 video base_url。")
elif not video_model_name:
    print("跳过视频理解示例：缺少视频理解模型名。")
else:
    video_demo_model = OpenAIModel(
        model=os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL"),
        video_api_key=video_api_key,
        video_base_url=video_base_url,
        video_model=video_model_name,
    )
    video_result = await video_demo_model.analyze_video(
        prompt="请用中文概括这个视频的主要内容，并按时间顺序说明关键动作。",
        video_path=video_demo_path,
        model=video_model_name,
        max_tokens=400,
    )
    print("视频理解模型：", video_model_name)
    print("视频路径：", video_demo_path)
    print("模型输出：", video_result.content)
    await video_demo_model.aclose()


### 方式 D：直接把 provider 配置字典交给 `create_agent(...)`

如果你不想先手写 `OpenAIModel(...)` 对象，也可以把配置字典直接传给 `create_agent(model={...})`。

这会走库内部的 `create_model_adapter(...)` 工厂。适合：

- 配置来自数据库 / YAML / 前端表单
- 你想把“用户自定义模型配置”当成纯数据存起来

In [ ]:
import os
import agentorch
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

dict_model_payload = {
    "provider": "openai",
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "timeout": 45.0,
    "max_retries": 1,
}
print("字典方式的 provider 配置：")
print(preview_model_config(dict_model_payload))

dict_agent = agentorch.create_agent(
    model=dict_model_payload,
    system_prompt="你是一个由配置字典驱动的助手。",
    name="dict-config-agent",
)
print("dict_agent blueprint kind：", dict_agent.export_blueprint()["kind"])
await dict_agent.aclose()


### 方式 E：接入 OpenAI 兼容 HTTP 服务

如果你的服务不是官方 OpenAI SDK 直连，而是任意一个 OpenAI 兼容 HTTP 网关，可以用 `OpenAICompatibleHTTPModel(...)`。

这条路径可以额外控制：

- `endpoint_path`
- `auth_scheme`
- `headers`

适合对接代理网关、私有化中转、特殊认证头。

In [ ]:
import os
from agentorch import OpenAICompatibleHTTPModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

openai_http_model = OpenAICompatibleHTTPModel(
    model="deepseek-v4-flash",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    endpoint_path="/chat/completions",
    auth_scheme="Bearer",
    headers={"X-Demo-Client": "masarch-notebook"},
)
print("OpenAI 兼容 HTTP 模型配置：")
print(preview_model_config(openai_http_model.config))
await openai_http_model.aclose()


### 方式 F：注册你自己的 provider

如果你希望用户在前端 / 配置文件里直接写：

```python
{
    "provider": "my_lab_gateway",
    "model": "research-chat-v1",
    ...
}
```

那么可以先注册一个 provider 名称，再把配置字典交给 `create_agent(...)` 或 `create_model_adapter(...)`。

In [ ]:
import os
import agentorch
from agentorch import OpenAICompatibleHTTPModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

CUSTOM_PROVIDER_NAME = "demo_custom_http_provider"


def build_demo_custom_provider(config: ModelConfig):
    return OpenAICompatibleHTTPModel.from_config(
        config,
        endpoint_path=config.endpoint_path or "/chat/completions",
        auth_scheme=config.auth_scheme or "Bearer",
    )


if CUSTOM_PROVIDER_NAME not in agentorch.list_model_providers():
    agentorch.register_model_provider(CUSTOM_PROVIDER_NAME, build_demo_custom_provider)

custom_provider_payload = {
    "provider": CUSTOM_PROVIDER_NAME,
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "endpoint_path": "/chat/completions",
    "headers": {"X-Provider-Name": CUSTOM_PROVIDER_NAME},
}

custom_provider_adapter = agentorch.create_model_adapter(custom_provider_payload)
print("自定义 provider 配置：")
print(preview_model_config(custom_provider_adapter.config))
await custom_provider_adapter.aclose()


### 用户客制化 model 的推荐落地方式

如果你要在自己的产品里让用户自定义模型，推荐把用户输入保存成一个结构化配置字典，而不是只存一个模型名。

推荐字段至少包括：

- `provider`
- `model`
- `api_key`
- `base_url`
- 可选 `endpoint_path`
- 可选 `embedding_api_key` / `embedding_base_url` / `embedding_model`
- 可选 `image_api_key` / `image_base_url` / `image_model`
- 可选 `video_api_key` / `video_base_url` / `video_model`

这样做的好处是：

- 前端表单、数据库、YAML、Notebook 都能共用同一份配置结构
- 可以很自然地支持“聊天一套供应商，图片另一套供应商”
- 后端只需要把这个字典交给 `create_agent(model=payload)` 或 `create_model_adapter(payload)`

In [ ]:
user_defined_model_payload = {
    "provider": "openai_http",
    "model": "deepseek-v4-flash",
    "api_key": "<用户自己的聊天 key>",
    "base_url": "https://your-gateway.example.com/v1",
    "embedding_api_key": "<用户自己的 embedding key>",
    "embedding_base_url": "https://your-embedding.example.com/v1",
    "embedding_model": "text-embedding-3-small",
    "image_api_key": "<用户自己的图片 key>",
    "image_base_url": "https://your-image.example.com/v1",
    "image_model": "image-model-name",
}

print("推荐给用户保存的 model payload 结构：")
print(user_defined_model_payload)


### 多智能体里，每个角色都可以绑定独立模型

如果你在做多智能体系统，完全可以让：

- `planner` 走一个供应商
- `reviewer` 走另一个供应商
- `supervisor` 再走第三个供应商或默认模型

也就是说，**每个 agent 都可以拥有自己独立的 `model/api_key/base_url`。**

推荐做法是：每个角色存一份自己的 `model payload`，然后分别构造 agent。

In [ ]:
import os
import agentorch
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

planner_model_payload = {
    "provider": "openai",
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
}

reviewer_model_payload = {
    "provider": "openai_http",
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "headers": {"X-Agent-Role": "reviewer"},
}

supervisor_model_payload = {
    "provider": "openai",
    "model": os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL"),
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
}

planner_agent = agentorch.create_agent(
    model=planner_model_payload,
    reasoning="plan_execute",
    name="custom-planner",
)
reviewer_agent = agentorch.create_agent(
    model=reviewer_model_payload,
    reasoning="react",
    name="custom-reviewer",
)
custom_team = agentorch.create_multi_agent(
    model=supervisor_model_payload,
    agents=[
        {"agent": planner_agent, "name": "planner", "role": "planner"},
        {"agent": reviewer_agent, "name": "reviewer", "role": "reviewer"},
    ],
    system_prompt="协调不同模型的专家并输出最终答案。",
    name="custom-model-team",
)

print("planner 配置：", preview_model_config(planner_model_payload))
print("reviewer 配置：", preview_model_config(reviewer_model_payload))
print("supervisor 配置：", preview_model_config(supervisor_model_payload))
print("custom_team kind：", custom_team.export_blueprint()["kind"])

await custom_team.aclose()


## 2.6 RAG 与知识库构建

如果你希望这个库不只是调用模型，还能基于本地资料回答问题，那么最常见的路径就是：

1. 准备本地文件或文档对象
2. 构建知识库（`IndexedKnowledgeBase` / `InMemoryKnowledgeBase`）
3. 给 agent 打开 `enable_rag=True`
4. 通过 `knowledge_paths`、`knowledge_base`、`knowledge_scope`、`rag` 控制检索行为

下面先用 notebook 里自动生成的测试文档，演示知识库构建、检索、RAG agent 运行、以及多智能体共享知识库。

In [ ]:
import os
from pathlib import Path

RAG_DEMO_DIR = Path.cwd() / "artifacts" / "notebook_rag_demo"
RAG_DEMO_DIR.mkdir(parents=True, exist_ok=True)

rag_doc_1 = RAG_DEMO_DIR / "agentorch_overview.txt"
rag_doc_2 = RAG_DEMO_DIR / "rag_notes.txt"

rag_doc_1.write_text(
    (
        "AgentTorch is imported as agentorch even when the package name is masarch.\n"
        "It supports single-agent orchestration, multi-agent coordination, tool calling, workflows, and RAG.\n"
        "Notebook demos should prefer explicit thread_id and self-contained cells."
    ),
    encoding="utf-8",
)
rag_doc_2.write_text(
    (
        "RAG can be enabled with enable_rag=True and knowledge_paths=[...].\n"
        "IndexedKnowledgeBase can ingest local files and expose a retriever.\n"
        "Knowledge scopes let different agents retrieve different subsets of documents."
    ),
    encoding="utf-8",
)

print("RAG 测试目录：", RAG_DEMO_DIR)
print("测试文件：", [rag_doc_1.name, rag_doc_2.name])


### 方式 A：直接构建 `IndexedKnowledgeBase`

这是最贴近真实项目的知识库对象。你可以把本地路径列表交给它，它会做 ingestion，然后暴露 retriever。

In [ ]:
import agentorch

if "rag_doc_1" not in globals() or "rag_doc_2" not in globals():
    raise RuntimeError("请先运行上一格，先创建 RAG 测试文档。")

rag_kb = await agentorch.IndexedKnowledgeBase.acreate(
    paths=[rag_doc_1, rag_doc_2],
    scopes=["notebook-demo", "agentorch-docs"],
)

print("知识库类型：", rag_kb.__class__.__name__)
print("已收录文档：", list(rag_kb.documents.keys()))
print("已收录资产：", list(rag_kb.assets.keys()))


### 方式 B：直接测试 retriever 检索结果

这一格不走大模型，直接测试知识库检索本身是否工作正常。适合先验证资料 ingestion 是否成功。

In [ ]:
from agentorch.knowledge import RetrievalQuery

if "rag_kb" not in globals():
    raise RuntimeError("请先运行上一格，先构建 rag_kb。")

retriever = rag_kb.get_retriever()
retrieved_chunks = await retriever.retrieve(
    RetrievalQuery(
        query="How do I enable RAG in AgentTorch?",
        top_k=3,
        scopes=["notebook-demo"],
    )
)

print("检索命中数量：", len(retrieved_chunks))
for index, item in enumerate(retrieved_chunks, start=1):
    print(f"[{index}] doc=", item.chunk.document_id)
    print(item.chunk.text[:160])


### 方式 C：让 agent 直接用 `knowledge_paths` 构建 RAG

这条路径对最终用户最友好：你不一定要先手工创建知识库对象，直接在 `create_agent(...)` 里提供：

- `enable_rag=True`
- `knowledge_paths=[...]`
- 可选 `knowledge_scope=[...]`
- 可选 `rag=...`


In [ ]:
import agentorch
from agentorch.knowledge import RagStrategyConfig

if "rag_doc_1" not in globals() or "rag_doc_2" not in globals():
    raise RuntimeError("请先运行 RAG 测试文档创建单元。")
if "live_model" not in globals() or "DummyModel" not in globals():
    raise RuntimeError("请先运行前面的真实模型/离线模型初始化单元。")

rag_enabled_agent = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(reply="rag fallback ok")),
    system_prompt="你是一个严格基于知识库回答的助手。",
    reasoning="react",
    enable_rag=True,
    knowledge_paths=[rag_doc_1, rag_doc_2],
    knowledge_scope=["notebook-demo"],
    rag=RagStrategyConfig.for_classic(top_k=3, injection_policy="summary_only"),
    name="rag-enabled-agent",
)

print("RAG agent blueprint kind：", rag_enabled_agent.export_blueprint()["kind"])
print("RAG agent 默认 knowledge_scope：", rag_enabled_agent.runtime.config.default_knowledge_scope)

if live_model is None:
    print("当前未绑定真实 LLM，已验证 RAG agent 组装成功；如需真实问答，请先保证 live_model 可用。")
else:
    rag_result = await rag_enabled_agent.run(
        "根据知识库说明：如何开启 RAG，以及这个库的导入名是什么？",
        thread_id="nb-import-quickstart-rag-001",
    )
    print("RAG 输出：", rag_result.output_text)
    print("RAG tokens：", rag_result.usage.total_tokens)

await rag_enabled_agent.aclose()


### 方式 D：显式传入 `knowledge_base`

如果你已经在系统里预先构建好了知识库对象，就直接传 `knowledge_base=rag_kb`。这比每次都重新 ingest `knowledge_paths` 更适合服务端复用。

In [ ]:
import agentorch
from agentorch.knowledge import RagStrategyConfig

if "rag_kb" not in globals():
    raise RuntimeError("请先运行知识库构建单元，先得到 rag_kb。")
if "live_model" not in globals() or "DummyModel" not in globals():
    raise RuntimeError("请先运行前面的真实模型/离线模型初始化单元。")

shared_kb_agent = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(reply="shared kb fallback ok")),
    system_prompt="你优先引用共享知识库。",
    reasoning="react",
    enable_rag=True,
    knowledge_base=rag_kb,
    knowledge_scope=["agentorch-docs"],
    rag=RagStrategyConfig.for_hybrid(top_k=2, injection_policy="summary_only"),
    name="shared-kb-agent",
)

print("共享知识库 agent 已创建：", shared_kb_agent.export_blueprint()["kind"])
print("共享知识库 agent scope：", shared_kb_agent.runtime.config.default_knowledge_scope)

if live_model is not None:
    shared_kb_result = await shared_kb_agent.run(
        "知识库里提到这个库支持哪些核心能力？",
        thread_id="nb-import-quickstart-rag-002",
    )
    print("共享知识库输出：", shared_kb_result.output_text)

await shared_kb_agent.aclose()


### 方式 E：多智能体共享知识库

多智能体系统里，也可以让多个成员共享同一份知识库，但给不同角色分配不同的 `knowledge_scope`。

In [ ]:
import agentorch

if "rag_kb" not in globals():
    raise RuntimeError("请先运行知识库构建单元，先得到 rag_kb。")
if "live_model" not in globals() or "DummyModel" not in globals():
    raise RuntimeError("请先运行前面的真实模型/离线模型初始化单元。")

rag_planner = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(name="rag-planner", reply="rag planner ready")),
    reasoning="plan_execute",
    name="rag-planner",
)
rag_reviewer = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(name="rag-reviewer", reply="rag reviewer ready")),
    reasoning="react",
    name="rag-reviewer",
)

rag_team = agentorch.create_multi_agent(
    model=(live_model if live_model is not None else DummyModel(name="rag-supervisor", reply="rag supervisor ready")),
    agents=[
        {"agent": rag_planner, "name": "planner", "role": "planner", "knowledge_scope": ["notebook-demo"]},
        {"agent": rag_reviewer, "name": "reviewer", "role": "reviewer", "knowledge_scope": ["agentorch-docs"]},
    ],
    shared_knowledge={
        "knowledge_base": rag_kb,
        "knowledge_scope": ["notebook-demo", "agentorch-docs"],
    },
    system_prompt="结合共享知识库协调多个角色。",
    name="rag-team",
)

print("RAG team kind：", rag_team.export_blueprint()["kind"])
print("RAG team members：", [member["name"] for member in rag_team.export_blueprint()["members"]])
print("shared knowledge scope：", rag_team.runtime.config.default_knowledge_scope)

RUN_RAG_TEAM_LIVE = False  # 多智能体共享知识库这条链更适合二次验证，默认先做结构验证。
if live_model is not None and RUN_RAG_TEAM_LIVE:
    rag_team_result = await rag_team.run(
        "结合知识库，给出这个库的导入名、RAG 开启方法，并由 reviewer 做一次简短复核。",
        thread_id="nb-import-quickstart-rag-003",
    )
    print("RAG team 输出：", rag_team_result.output_text)
elif live_model is not None:
    print("已完成多智能体共享知识库的结构验证；如需真实运行，把 RUN_RAG_TEAM_LIVE 改成 True。")
else:
    print("当前未绑定真实 LLM，已完成多智能体共享知识库的结构验证。")

await rag_team.aclose()


## 3.1 记忆力与多轮线程

这个库不只是“单轮问答”。它支持：

- 基于 `thread_id` 的多轮会话
- `MemoryManager` 统一管理线程消息、agent 本地记忆、workspace 记录、共享笔记、collective memory
- 在多智能体场景里临时挂接共享记忆

下面先做两类演示：

1. 直接调用 `MemoryManager` 的底层接口
2. 用同一个 `thread_id` 跑 agent，观察多轮上下文行为

In [ ]:
import agentorch
from agentorch.agents import SharedNote, TaskArtifact
from agentorch.config import MemoryConfig
from agentorch.core import Message

MEMORY_DEMO_DIR = Path.cwd() / "artifacts" / "notebook_memory_demo"
MEMORY_DEMO_DIR.mkdir(parents=True, exist_ok=True)

memory_manager = agentorch.MemoryManager.compose(
    "session_memory",
    "thread_summary_memory",
    "agent_local_memory",
    "workspace_memory",
    "shared_note_memory",
    "record_memory",
    "collective_memory",
    config=MemoryConfig(
        checkpoint_path=MEMORY_DEMO_DIR / "checkpoints.db",
        record_path=MEMORY_DEMO_DIR / "records.db",
    ),
)

memory_thread_id = "nb-memory-demo-001"
await memory_manager.append_message(memory_thread_id, Message(role="user", content="hello memory"))
await memory_manager.append_message(memory_thread_id, Message(role="assistant", content="memory acknowledged"))

messages = await memory_manager.get_thread_messages(memory_thread_id)
summary = await memory_manager.summarize_thread(memory_thread_id)
await memory_manager.append_agent_memory(memory_thread_id, "coder", {"goal": "remember notebook task"})
agent_memory = await memory_manager.get_agent_memory(memory_thread_id, "coder")

workspace_record = await memory_manager.write_workspace_record(
    memory_thread_id,
    task_id="memory-task-1",
    owner_agent="coder",
    artifact=TaskArtifact(name="draft-note", content="artifact from notebook memory demo"),
)
shared_note = SharedNote(
    note_id="note-1",
    task_id="memory-task-1",
    author_agent="coder",
    content="candidate insight for notebook demo",
    metadata={"collective_candidate": True},
)
await memory_manager.add_shared_note(memory_thread_id, shared_note)
collective_id = await memory_manager.promote_collective_memory(
    thread_id=memory_thread_id,
    kind="lesson",
    content="validated notebook memory insight",
    tags=["memory", "notebook"],
    source_agents=["coder"],
)
collective_records = await memory_manager.search_collective_memory(query="validated", thread_id=memory_thread_id)

print("记忆机制：", memory_manager.list_mechanisms())
print("线程消息：", [message.content for message in messages])
print("线程摘要：", summary)
print("agent 本地记忆：", agent_memory)
print("workspace artifact：", workspace_record.name)
print("collective memory 命中数：", len(collective_records))
print("collective memory id：", collective_id)


In [ ]:
memory_demo_agent = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(reply="memory demo ok")),
    system_prompt="你需要记住同一 thread_id 下的上下文。",
    reasoning="react",
    memory=memory_manager,
    name="memory-demo-agent",
)

memory_agent_thread_id = "nb-memory-agent-001"
if live_model is not None:
    turn_1 = await memory_demo_agent.run(
        "我叫小李，请记住我的名字。",
        thread_id=memory_agent_thread_id,
    )
    turn_2 = await memory_demo_agent.run(
        "我刚才告诉你的名字是什么？",
        thread_id=memory_agent_thread_id,
    )
    print("第1轮输出：", turn_1.output_text)
    print("第2轮输出：", turn_2.output_text)
else:
    print("当前未绑定真实 LLM，已验证 memory agent 组装成功。")

await memory_demo_agent.aclose()


## 3.2 结构化输出与解析器

如果你需要把 agent 输出落成 JSON、列表、键值对或 Pydantic 对象，这个库已经内置了 parser 体系。

In [ ]:
from pydantic import BaseModel


class NotebookAnswer(BaseModel):
    summary: str


json_parser = agentorch.JSONParser()
list_parser = agentorch.ListParser()
kv_parser = agentorch.KeyValueParser()
pydantic_parser = agentorch.PydanticParser(NotebookAnswer)

print("parse_json：", await json_parser.parse('{"name": "agentorch"}'))
print("parse_list：", await list_parser.parse('- alpha\n- beta'))
print("parse_key_values：", await kv_parser.parse('x: 1\ny=2'))
print("parse_pydantic：", await pydantic_parser.parse('{"summary": "notebook parser demo"}'))

parser_chain = agentorch.parser_chain(agentorch.JSONParser(), agentorch.TextParser())
parsed_json = await parser_chain.parse('{"ok": true}')
print("parser_chain 输出：", parsed_json)
print("format_prompt 示例：", agentorch.format_prompt('Return a JSON object', agentorch.JSONParser())[:120], '...')


In [ ]:
if live_model is not None:
    parsed_agent = agentorch.create_agent(
        model=live_model,
        system_prompt="Return concise structured output.",
        reasoning="react",
        name="parsed-agent",
    )
    parsed_result = await parsed_agent.run_parsed(
        "请输出一个 JSON，对象里包含 key=summary，值为一句中文总结。",
        thread_id="nb-parsed-agent-001",
        parser=agentorch.JSONParser(),
    )
    print("原始输出：", parsed_result.raw.output_text)
    print("解析结果：", parsed_result.parsed)
    await parsed_agent.aclose()
else:
    print("当前未绑定真实 LLM，结构化输出的本地 parser 示例已覆盖核心用法。")


## 3.3 Workflow / DAG

当任务顺序必须显式固定时，可以把步骤组织成 workflow。下面先展示节点构建和 DAG 组装。

In [ ]:
from agentorch.workflow import Node, WorkflowBuilder

retrieve_node = Node.retrieve("retrieve", question="what is agentorch", output_key="retrieved")
mount_node = Node.rag_mount("mount", from_variable="retrieved")
answer_node = Node.model_node("answer", prompt="Answer the user with mounted retrieval context.")
aggregate_node = Node.aggregate("aggregate", sources=["retrieved"], output_key="combined")

workflow = (
    WorkflowBuilder(max_steps=6)
    .add(retrieve_node, entry=True)
    .add(mount_node)
    .connect("retrieve", "mount")
    .then(answer_node)
    .then(aggregate_node)
    .build()
)

print("workflow entry：", workflow.entry_node)
print("workflow nodes：", [node.id for node in workflow.nodes])
print("workflow edges：", [(edge.source, edge.target, edge.kind) for edge in workflow.edges])


## 3.4 Streaming 事件流

这个库支持 `stream=True`。在 notebook 里要用：

```python
async for event in agent.run(..., stream=True):
    ...
```

In [ ]:
streaming_agent = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(reply="streaming demo ok")),
    system_prompt="Stream concise answers.",
    reasoning="react",
    enable_streaming=True,
    name="streaming-agent",
)

if live_model is not None:
    stream_events = []
    preview_limit = 12
    async for event in streaming_agent.run(
        "请用一句话介绍 agentorch。",
        thread_id="nb-stream-001",
        stream=True,
    ):
        stream_events.append(event)
        if len(stream_events) <= preview_limit:
            print("stream event：", event.event_type)
    print("stream 事件总数：", len(stream_events))
    if stream_events:
        print("最后一个事件类型：", stream_events[-1].event_type)
        if stream_events[-1].result is not None:
            print("最终输出：", stream_events[-1].result.output_text)
else:
    print("当前未绑定真实 LLM，streaming 入口已验证可构造。")

await streaming_agent.aclose()


## 3.5 Observability 与事件存储

这个库可以把运行事件写入 SQLite，并带有默认脱敏。下面先做底层事件存储示例。

In [ ]:
from agentorch.config import ObservabilityConfig
from agentorch import PayloadBudgetConfig, RedactionConfig
from agentorch.observability import SQLiteEventStore

OBS_DEMO_DIR = Path.cwd() / "artifacts" / "notebook_observability_demo"
OBS_DEMO_DIR.mkdir(parents=True, exist_ok=True)
event_store = SQLiteEventStore(
    OBS_DEMO_DIR / "observability.db",
    redaction=RedactionConfig(),
    payload_budget=PayloadBudgetConfig(max_total_chars=300, max_string_chars=60, max_collection_items=2),
)
event_store.emit(
    "human_feedback_emitted",
    {
        "run_id": "run-notebook-001",
        "thread_id": "thread-notebook-001",
        "message": "token sk-secret-notebook-key should not be persisted",
        "metadata": {"api_key": "plain-secret", "items": list(range(10))},
    },
)
stored_events = event_store.get_run_events("run-notebook-001")
print("observability 事件数：", len(stored_events))
print("observability 首条事件：", stored_events[0])


## 3.6 更多可继续扩展的能力面

到这里，这个 notebook 已经覆盖了：

- 单智能体
- 工具调用
- 模型客制化
- RAG / 知识库
- 多智能体
- 记忆能力
- 结构化输出
- workflow
- streaming
- observability

如果还要继续扩展，下一层最值得加的是：

- `skills` 显式加载与路由
- `design` / `compose_agent` / `compose_team`
- `evolution` 搜索与候选体构建
- `sandbox` 与执行工具
- `human_feedback` 与 HITL

## 3. 多智能体示例

如果你已经能成功导入并跑通前两格，这一格可以继续确认 `create_multi_agent(...)` 的基本用法。

In [ ]:
use_dummy = False  # 默认不回退到离线演示，避免把假模型误当真实 LLM
if live_model is None and not use_dummy:
    print("多智能体示例跳过：当前没有真实 LLM；如需离线演示，把 use_dummy 改成 True。")
else:
    planner = agentorch.create_agent(
        model=(live_model if live_model is not None else DummyModel(name="planner-model", reply="planner ready")),
        reasoning="plan_execute",
        name="planner",
    )

    reviewer = agentorch.create_agent(
        model=(live_model if live_model is not None else DummyModel(name="reviewer-model", reply="reviewer ready")),
        reasoning="react",
        name="reviewer",
    )

    team = agentorch.create_multi_agent(
        model=(live_model if live_model is not None else DummyModel(name="supervisor-model", reply="supervisor ready")),
        agents=[
            {"agent": planner, "name": "planner", "role": "planner"},
            {"agent": reviewer, "name": "reviewer", "role": "reviewer"},
        ],
        system_prompt="Coordinate specialists and return one final answer.",
        name="demo-team",
    )

    team_result = await team.run(
        "Draft and review a migration plan.",
        thread_id="nb-import-quickstart-003",
    )

    print("team 输出：", team_result.output_text)
    print("team kind：", team.export_blueprint()["kind"])

    await team.aclose()


## 4. 可选：真实模型示例

如果你已经在当前 notebook 内核里设置了下面这些环境变量：

- `OPENAI_API_KEY`
- `OPENAI_CHAT_MODEL` 或 `OPENAI_MODEL` 或 `AGENTORCH_MODEL`
- 如果不是官方接口，再补 `OPENAI_BASE_URL`

就可以运行这一格。否则它会自动跳过。

In [ ]:
if live_model is not None:
    live_agent = agentorch.create_agent(
        model=live_model,
        system_prompt="You are concise.",
        reasoning="react",
    )

    live_result = await live_agent.run(
        "用一句中文介绍你自己。",
        thread_id="nb-import-quickstart-004",
    )

    print("真实模型输出：", live_result.output_text)
    print("真实模型 tokens：", live_result.usage.total_tokens)
    await live_agent.aclose()
else:
    print("跳过真实模型示例：当前内核没有同时提供 OPENAI_API_KEY 和模型名。")


## 5. 结论

如果你只记一件事，就记这个：

```python
# 安装
pip install masarch==0.1.0

# 导入
import agentorch
```

也就是：**装 `masarch`，导 `agentorch`**。